# Matching & Analysis Engine
This notebook loads the ingredient database and matches OCR-extracted ingredients against it using fuzzy matching. Flags harmful ingredients and generates a safety analysis by skin type.

In [1]:
import pandas as pd
from rapidfuzz import process, fuzz
import json

In [2]:
db = pd.read_csv('C:/Users/anamt/OneDrive/Desktop/skincare-analyzer/data/ingredients_database.csv', encoding='latin-1')

print(f"Database loaded: {len(db)} ingredients")
print(db.columns.tolist())
print()
print(db.head(3))

Database loaded: 385 ingredients
['ingredient_name', 'common_aliases', 'category', 'concern_level', 'explanation', 'skin_types_to_avoid']

           ingredient_name                                     common_aliases  \
0    Sodium Lauryl Sulfate  SLS, Sodium Laurilsulfate, Sodium Dodecyl Sulfate   
1   Sodium Laureth Sulfate  SLES, Sodium Lauryl Ether Sulfate, Sodium Poly...   
2  Ammonium Lauryl Sulfate                      ALS, Ammonium Dodecyl Sulfate   

  category concern_level                                        explanation  \
0  Sulfate         Avoid  Harsh surfactant that strips the skin's natura...   
1  Sulfate       Caution  Milder than SLS but still a strong surfactant....   
2  Sulfate         Avoid  Similar to SLS in irritation potential. Strips...   

                  skin_types_to_avoid  
0  sensitive, dry, acne-prone, eczema  
1              sensitive, dry, eczema  
2          sensitive, dry, acne-prone  


In [3]:
search_terms = []

for idx, row in db.iterrows():
    # Add the ingredient name itself
    search_terms.append((row['ingredient_name'].lower().strip(), idx))
    
    # Add each alias separately
    if pd.notna(row['common_aliases']):
        for alias in row['common_aliases'].split(','):
            alias = alias.lower().strip()
            if alias:
                search_terms.append((alias, idx))

print(f"Total searchable terms: {len(search_terms)}")
print("Sample:", search_terms[:5])

Total searchable terms: 925
Sample: [('sodium lauryl sulfate', 0), ('sls', 0), ('sodium laurilsulfate', 0), ('sodium dodecyl sulfate', 0), ('sodium laureth sulfate', 1)]


In [4]:
terms_only = [term for term, idx in search_terms]

def match_ingredient(ingredient, threshold=80):
    """
    Takes one ingredient name from OCR output.
    Returns the best matching database row or None if no good match found.
    """
    # Find the best match from all 925 searchable terms
    result = process.extractOne(ingredient, terms_only, scorer=fuzz.token_sort_ratio)
    
    if result is None:
        return None
    
    matched_term, score, term_index = result
    
    # Only accept match if score is above threshold
    if score < threshold:
        return None
    
    # Get the database row index from search_terms
    db_index = search_terms[term_index][1]
    
    # Return the full database row as a dictionary
    return db.iloc[db_index].to_dict()

In [5]:
def analyze_ingredients(ingredient_list, skin_type=None):
    """
    Takes a list of ingredients from OCR output.
    Returns a full analysis with flagged ingredients and safety summary.
    """
    results = {
        'matched': [],      # ingredients found in database
        'not_found': [],    # ingredients not in database
        'flagged': [],      # ingredients to be concerned about
        'safe_count': 0,
        'caution_count': 0,
        'avoid_count': 0
    }
    
    for ingredient in ingredient_list:
        match = match_ingredient(ingredient)
        
        if match is None:
            results['not_found'].append(ingredient)
            continue
        
        # Add to matched list
        results['matched'].append(match)
        
        # Count concern levels
        if match['concern_level'] == 'Safe':
            results['safe_count'] += 1
        elif match['concern_level'] == 'Caution':
            results['caution_count'] += 1
        elif match['concern_level'] == 'Avoid':
            results['avoid_count'] += 1
        
        # Flag if concern level is not Safe
        if match['concern_level'] in ['Caution', 'Avoid']:
            # If skin type provided, check if this ingredient is problematic for them
            if skin_type:
                skin_types_to_avoid = str(match['skin_types_to_avoid']).lower()
                if skin_type.lower() in skin_types_to_avoid:
                    results['flagged'].append(match)
            else:
                results['flagged'].append(match)
    
    return results

In [6]:
def calculate_safety_score(results):
    """
    Calculates an overall safety score out of 100.
    Avoid ingredients heavily penalize the score.
    Caution ingredients moderately penalize.
    """
    total = len(results['matched'])
    
    if total == 0:
        return 0

    # Count concern levels from flagged list only
    avoid_count = sum(1 for item in results['flagged'] if item['concern_level'] == 'Avoid')
    caution_count = sum(1 for item in results['flagged'] if item['concern_level'] == 'Caution')
    
    # Each Avoid ingredient deducts 10 points
    # Each Caution ingredient deducts 3 points
    deductions = (avoid_count * 10) + (caution_count * 3)
    
    score = max(0, 100 - deductions)
    
    return score

In [7]:
def generate_report(ingredient_list, skin_type=None):
    """
    Runs full analysis and prints a clean summary report.
    """
    results = analyze_ingredients(ingredient_list, skin_type)
    score = calculate_safety_score(results)
    
    print("=" * 50)
    print("SKINCARE INGREDIENT ANALYSIS REPORT")
    print("=" * 50)
    if skin_type:
        print(f"Skin Type: {skin_type.capitalize()}")
    print(f"Total Ingredients: {len(ingredient_list)}")
    print(f"Matched in Database: {len(results['matched'])}")
    print(f"Safety Score: {score}/100")
    print()
    
    print(f"Safe: {results['safe_count']}  |  Caution: {results['caution_count']}  |  Avoid: {results['avoid_count']}")
    print()
    
    if results['flagged']:
        print("⚠ FLAGGED INGREDIENTS:")
        for item in results['flagged']:
            print(f"  [{item['concern_level']}] {item['ingredient_name']}")
            print(f"  {item['explanation'][:100]}...")
            print()
    else:
        print("✓ No flagged ingredients for your skin type.")
    
    if results['not_found']:
        print(f"? NOT IN DATABASE ({len(results['not_found'])} ingredients):")
        for item in results['not_found']:
            print(f"  - {item}")
    
    print("=" * 50)
    
    return results, score

In [9]:
# Test with real OCR output from Wishcare Sunscreen
wishcare_ingredients = [
    "cyclopentasiloxane", "octyl salicylate", "niacinamide", "silica",
    "avobenzone", "octocrylene", "butylene glycol", "glycerine", "homosalate",
    "tapioca starch", "neopentyl glycol diheptanoate", "octyldodecanol",
    "polysorbate-20", "zinc oxide", "titanium dioxide", "oats extract",
    "zinc pca", "ceramide ap", "ceramide np", "ceramide eos", "d-panthenol",
    "centella asiatica extract", "ethylhexyl glycerine", "phenoxyethanol",
    "coco-glucoside", "xanthan gum", "tocopherol acetate", "hyaluronic acid",
    "citric acid", "aqua"
]

generate_report(wishcare_ingredients, skin_type="sensitive");

SKINCARE INGREDIENT ANALYSIS REPORT
Skin Type: Sensitive
Total Ingredients: 30
Matched in Database: 30
Safety Score: 82/100

Safe: 21  |  Caution: 9  |  Avoid: 0

⚠ FLAGGED INGREDIENTS:
  [Caution] Octyl Salicylate
  Chemical UV filter used in sunscreens. Weak UVB absorber. Can cause irritation in sensitive skin. Av...

  [Caution] Avobenzone
  Chemical UV filter with some penetration concerns. Degrades in sunlight and needs stabilisers. Gener...

  [Caution] Octocrylene
  Chemical UV filter used in sunscreens. Can cause contact allergies in sensitive individuals. May deg...

  [Caution] Polysorbate 20
  Emulsifier and solubilizer commonly used in toners and mists. Generally safe but can cause irritatio...

  [Caution] Phenoxyethanol
  Widely used preservative considered safer than parabens. Some concerns about irritation at higher co...

  [Caution] Citric Acid
  Mild AHA used primarily as a pH adjuster. At higher concentrations can exfoliate but also irritate....

